# Step 11.3 — `QK^T` as Compatibility Score：Query 怎样判断哪个 Key 更匹配？

Step 11.2 已经得到：

$$
Q\in\mathbb R^{B\times T\times d_k},
\qquad
K\in\mathbb R^{B\times T\times d_k}.
$$

现在第一次回答：

> **每个 Query position 怎样同时给所有 Key positions 一个 relevance / compatibility score？**

最基本的做法是 dot product（点积）。把所有 positions 一次性写成矩阵运算，就是 `QK^T`。

本节只学习 dot product、它与 cosine similarity 的区别，以及 `(B,T,T)` score matrix 的含义；不做 scaling，不做 softmax，不进入 Step 11.4。


## 1. Dot Product：两个 d_k 维向量怎样变成一个 Scalar Score？

先只看一个 Query vector 与一个 Key vector：

$$
q=
\begin{bmatrix}
q_1&q_2&\cdots&q_{d_k}
\end{bmatrix},
$$

$$
k=
\begin{bmatrix}
k_1&k_2&\cdots&k_{d_k}
\end{bmatrix}.
$$

它们的 **dot product（点积）** 定义为：

$$
\boxed{
q\cdot k
=
\sum_{m=1}^{d_k}q_mk_m
}
$$

输入是两个 `(d_k,)` vectors，输出是一个 scalar。

例如：

$$
q=[1,2],
\qquad
k=[3,1].
$$

那么：

$$
q\cdot k
=
1\times3+2\times1
=
5.
$$

在 Attention 里，这个 scalar 被用作一个 **compatibility score（兼容度分数）**：

```text
Query position i
      与
Key position j
      ↓
dot product
      ↓
score s_(i,j)
```

注意：这里的 score 还不是 probability，也不是 attention weight；它可以为正、为零、为负，也没有被限制到 `[0,1]`。


## 2. Dot Product 不等于 Cosine Similarity

这是一个必须现在就分清的边界。

Dot product：

$$
q\cdot k.
$$

Cosine similarity：

$$
\boxed{
\cos\theta
=
\frac{q\cdot k}
{\|q\|_2\|k\|_2}
}
$$

其中 `||q||_2` 与 `||k||_2` 是前面已经学过的 L2 norm。

Cosine similarity 把两个向量的长度除掉，主要比较方向；dot product 同时受到**方向和向量 magnitude（长度）**影响。

最小例子：

$$
q=[1,0],
\quad
k_1=[1,0],
\quad
k_2=[2,0].
$$

它们方向完全相同，因此：

$$
\cos(q,k_1)=1,
\qquad
\cos(q,k_2)=1.
$$

但 dot products 是：

$$
q\cdot k_1=1,
\qquad
q\cdot k_2=2.
$$

所以不能把 Attention 中的 raw `q·k` 直接叫 cosine similarity。

只有当 q 与 k 都已经单位归一化：

$$
\|q\|_2=\|k\|_2=1
$$

时，dot product 才数值上等于 cosine similarity。


In [ ]:
import mlx.core as mx

q = mx.array([1.0, 0.0])
k1 = mx.array([1.0, 0.0])
k2 = mx.array([2.0, 0.0])

dot1 = mx.sum(q * k1)
dot2 = mx.sum(q * k2)

cos1 = dot1 / (
    mx.sqrt(mx.sum(q * q)) * mx.sqrt(mx.sum(k1 * k1))
)
cos2 = dot2 / (
    mx.sqrt(mx.sum(q * q)) * mx.sqrt(mx.sum(k2 * k2))
)

print('dot(q,k1):', dot1)
print('dot(q,k2):', dot2)
print('cos(q,k1):', cos1)
print('cos(q,k2):', cos2)


## 3. 从一个 Query–Key Pair，扩展到所有 T×T Pairs

沿用 Step 11.2 的小例子：

$$
Q=
\begin{bmatrix}
1&2\\
1&0
\end{bmatrix},
\qquad
K=
\begin{bmatrix}
3&1\\
1&1
\end{bmatrix}.
$$

这里：

```text
T = 2
d_k = 2
```

我们希望得到四个 pairwise scores：

```text
q_0 with k_0
q_0 with k_1
q_1 with k_0
q_1 with k_1
```

分别是：

$$
q_0\cdot k_0
=
[1,2]\cdot[3,1]
=
5,
$$

$$
q_0\cdot k_1
=
[1,2]\cdot[1,1]
=
3,
$$

$$
q_1\cdot k_0
=
[1,0]\cdot[3,1]
=
3,
$$

$$
q_1\cdot k_1
=
[1,0]\cdot[1,1]
=
1.
$$

把它们按：

```text
rows    = Query positions i
columns = Key positions j
```

排成矩阵：

$$
S=
\begin{bmatrix}
5&3\\
3&1
\end{bmatrix}.
$$

也就是：

$$
\boxed{S_{i,j}=q_i\cdot k_j}
$$

这个例子中的 S 恰好是对称矩阵，只是这些教学数值造成的巧合。一般情况下 `Q != K`，所以 `QK^T` **不保证对称**。


## 4. 为什么矩阵公式正好是 `QK^T`？

先去掉 Batch。

有：

$$
Q:(T,d_k)
$$

以及：

$$
K:(T,d_k).
$$

我们真正想让 Query 的 feature axis `d_k` 与 Key 的 feature axis `d_k` 做 dot product，因此把 K 转置：

$$
K^T:(d_k,T).
$$

于是：

$$
\boxed{
S=QK^T
}
$$

shape：

$$
(T,d_k)(d_k,T)
\rightarrow
(T,T).
$$

矩阵乘法的定义正好自动完成所有 pairwise dot products。

### 行和列分别代表谁？

这是后续所有 Attention shape tracing 的关键：

$$
S_{i,j}=q_i\cdot k_j.
$$

因此：

```text
S 的第 i 行
= Query position i 对所有 Key positions 的 scores

S 的第 j 列
= 所有 Query positions 对 Key position j 的 scores
```

绝对不要把 row/column 角色反过来。


In [ ]:
Q = mx.array([
    [1.0, 2.0],
    [1.0, 0.0],
])  # (T,d_k)

K = mx.array([
    [3.0, 1.0],
    [1.0, 1.0],
])  # (T,d_k)

scores = Q @ K.T  # (T,T)

print('Q:', Q.shape)
print('K:', K.shape)
print('K.T:', K.T.shape)
print('scores:', scores)
print('scores shape:', scores.shape)


## 5. 加回 Batch：`(B,T,d_k) @ (B,d_k,T) → (B,T,T)`

真实 Self-Attention 中：

$$
Q\in\mathbb R^{B\times T\times d_k},
\qquad
K\in\mathbb R^{B\times T\times d_k}.
$$

只转置 K 的最后两个 axes：

$$
K^{T_{last2}}
\in
\mathbb R^{B\times d_k\times T}.
$$

MLX 可以写：

```python
K_transposed = mx.swapaxes(K, -1, -2)
```

于是 batched matrix multiplication：

$$
(B,T,d_k)
@
(B,d_k,T)
\rightarrow
\boxed{(B,T,T)}.
$$

每个 batch sample 独立得到一个：

$$
T\times T
$$

compatibility score matrix。

这个 `T×T` 结构非常重要：sequence 中每个 Query position，都对同一 sequence 的每个 Key position 有一个 raw score。

### 一个真正非对称的 sanity check

取：

$$
Q=
\begin{bmatrix}
1&0\\
0&2
\end{bmatrix},
\qquad
K=
\begin{bmatrix}
1&1\\
1&3
\end{bmatrix}.
$$

则：

$$
QK^T=
\begin{bmatrix}
1&1\\
2&6
\end{bmatrix}.
$$

这里明显：

$$
S_{0,1}=1
\neq
S_{1,0}=2.
$$

所以一般情况下 `QK^T` 不要求对称。只有特殊条件下，例如 `Q=K` 时得到 `QQ^T`，才一定对称。

### Step 11.3 材料边界

- 单个 compatibility score 是 `q_i·k_j = sum_m q_(i,m) k_(j,m)`，输出一个 scalar。
- dot product 同时受方向与 magnitude 影响；cosine similarity 会除以两个向量的 L2 norms。只有单位归一化后两者才数值相同。
- 把所有 pairwise Query–Key dot products 排成矩阵，就是 `S=QK^T`。
- 行 `i` 表示 Query position i，列 `j` 表示 Key position j，`S_(i,j)=q_i·k_j`。
- Batch 下 shape 是 `(B,T,d_k) @ (B,d_k,T) -> (B,T,T)`。
- scores 仍是未归一化 raw compatibility values，不是 attention probabilities。
- 本轮只完成 compatibility score；不进入 Step 11.4，不做 scaling 或 softmax。
